In [20]:
import xarray as xr
import numpy as np
from scipy.interpolate import interpn

In [36]:
range_id = 0


is_single_point = True
grid_point = (np.float64(-29.0), np.float64(150.0))
model_radius = 0.0
silo_radius = 0.25
n_points_inner_circle = 6
radius_inner_circle = 0.6
n_points_outer_circle = 6
radius_outer_circle = 1.04

In [37]:
ecmwf = xr.open_dataset(f"/scratch/tm70/ag9761/iceds_datasets/ecmwf_51_{range_id}_new_processed.nc", chunks="auto")
ecmwf = ecmwf.assign_coords(model='ecmwf')

ukmo = xr.open_dataset(f"/scratch/tm70/ag9761/iceds_datasets/ukmo_604_{range_id}_new_processed.nc", chunks="auto")
ukmo = ukmo.assign_coords(model='ukmo')

access = xr.open_dataset(f"/scratch/tm70/ag9761/iceds_datasets/bom_2_{range_id}_new_processed.nc", chunks="auto")
access = access.assign_coords(model='access')

silo = xr.open_dataset(f"/scratch/tm70/ag9761/iceds_datasets/silo_monthly_rain_iceds_{range_id}_processed.nc", chunks="auto")
silo = silo.assign_coords(model='silo')

In [38]:
def construct_circle(origin, n_points, radius, base_angle=0.0, include_origin=False):

    def get_circular_point(angle):
        print(angle)
        return (round(origin[0] + radius * xr.ufuncs.cos(angle), 2), round(origin[1] + radius * xr.ufuncs.sin(angle), 2))
    
    coords = [get_circular_point(base_angle + 2 * i * np.pi / n_points_inner_circle) for i in range(n_points)]
    if include_origin:
        coords.insert(len(coords) // 2, origin)

    return tuple(map(list, zip(*coords)))

In [39]:
# ecmwf["pr"] = ecmwf["pr"].transpose("latitude", "longitude", "time", "lead") 
# # temp = ecmwf.interp(latitude=-29.0, longitude=150.0, method="linear")
# # print(temp)
# lat = ecmwf.latitude.values
# lon = ecmwf.longitude.values
# values = ecmwf.pr.values
# result = interpn(
#     (lat, lon),
#     values,
#     np.array([[-29.0, 150.0], ]),
#     method="linear",
# )
# print(lon)

In [40]:
# result[:, :, 0]

In [41]:
silo

<xarray.Dataset> Size: 37MB
Dimensions:    (time: 288, latitude: 126, longitude: 127)
Coordinates:
  * time       (time) datetime64[ns] 2kB 1993-01-01 1993-02-01 ... 2016-12-01
  * latitude   (latitude) float64 1kB -26.15 -26.1 -26.05 ... -20.0 -19.95 -19.9
  * longitude  (longitude) float64 1kB 140.8 140.9 140.9 ... 147.1 147.1 147.2
    model      <U4 16B 'silo'
Data variables:
    pr         (time, latitude, longitude) float64 37MB dask.array<chunksize=(288, 126, 127), meta=np.ndarray>
Attributes: (12/16)
    CDI:                      Climate Data Interface version 2.4.3 (https://m...
    Conventions:              CF-1.6
    institution:              Queensland Government, Department of Environmen...
    department:               Department of Environment and Science
    department_short:         DES
    copyright:                Copyright - the State of Queensland Department ...
    ...                       ...
    reference:                Jeffrey, S.J., Carter, J.O., Moodie, K.B.A. and...
    disclaimer:               1. The user accepts all responsibility and risk...
    history:                  Wed Apr 22 17:32:41 2026: cdo mergetime 1993.mo...
    CDO:                      Climate Data Operators version 2.4.3 (https://m...
    standard_name:            time
    axis:                     T

In [42]:
preprocessed_dataset = xr.concat([ecmwf, ukmo, access], dim='model', coords="minimal")

/jobfs/172564815.gadi-pbs/ipykernel_3691425/2396936402.py:1: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  preprocessed_dataset = xr.concat([ecmwf, ukmo, access], dim='model', coords="minimal")


In [43]:
preprocessed_dataset.to_netcdf(f"/scratch/tm70/ag9761/iceds_datasets/preprocessed_dataset_{range_id}_new.nc")

In [10]:
preprocessed_dataset
# 1993-2015

<xarray.Dataset> Size: 62MB
Dimensions:    (model: 3, lead: 6, time: 293, latitude: 21, longitude: 141)
Coordinates:
  * model      (model) <U6 72B 'ecmwf' 'ukmo' 'access'
  * lead       (lead) int64 48B 0 1 2 3 4 5
  * time       (time) datetime64[ns] 2kB 1993-01-01 1993-02-01 ... 2017-05-01
  * latitude   (latitude) float64 168B -27.0 -27.05 -27.1 ... -27.9 -27.95 -28.0
  * longitude  (longitude) float64 1kB 148.0 148.1 148.1 ... 154.9 155.0 155.0
Data variables:
    pr         (model, lead, time, latitude, longitude) float32 62MB 2.572 .....
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts

In [7]:
ukmo

<xarray.Dataset> Size: 115kB
Dimensions:    (lead: 6, time: 292, latitude: 4, longitude: 4)
Coordinates:
  * lead       (lead) int64 48B 0 1 2 3 4 5
  * time       (time) datetime64[ns] 2kB 1993-02-01 1993-03-01 ... 2017-05-01
  * latitude   (latitude) float64 32B -22.0 -23.0 -24.0 -25.0
  * longitude  (longitude) float64 32B 142.0 143.0 144.0 145.0
    model      <U4 16B 'ukmo'
Data variables:
    pr         (lead, time, latitude, longitude) float32 112kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             egrr
    GRIB_centreDescription:  U.K. Met Office - Exeter
    GRIB_subCentre:          98
    Conventions:             CF-1.7
    institution:             U.K. Met Office - Exeter

In [8]:
access

<xarray.Dataset> Size: 115kB
Dimensions:    (lead: 6, time: 293, latitude: 4, longitude: 4)
Coordinates:
  * lead       (lead) int64 48B 0 1 2 3 4 5
  * time       (time) datetime64[ns] 2kB 1993-01-01 1993-02-01 ... 2017-05-01
  * latitude   (latitude) float64 32B -22.0 -23.0 -24.0 -25.0
  * longitude  (longitude) float64 32B 142.0 143.0 144.0 145.0
    model      <U6 24B 'access'
Data variables:
    pr         (lead, time, latitude, longitude) float32 113kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ammc
    GRIB_centreDescription:  Melbourne
    GRIB_subCentre:          98
    Conventions:             CF-1.7
    institution:             Melbourne

In [ ]:
# https://gist.github.com/arbennett/df50755671dff0e32e43cd46b9f06c9e